# LLE with DryingOven

건조기의 복합 센서 데이터(예: 온도, 습도, 진동, 전류, 무게 등) 분석에 LLE를 적용하는 것은 매우 훌륭하고 실무에서 자주 쓰이는 접근 방식입니다.( (사)한국산학기술학회)

가전제품이나 산업용 건조기 시스템은 외부 환경, 세탁물의 양과 재질 등에 따라 내부 물리량이 단순한 직선 형태가 아닌 매우 복잡한 곡선(비선형) 형태로 변하기 때문입니다.

## 1. 건조기 복합 센서 데이터에서의 LLE 활용 시나리오

### 이상 탐지 및 고장 진단 (Fault Detection)

* 정상적인 건조 사이클에서 발생하는 센서 데이터들의 고유한 기하학적 형태(정상 매니폴드)를 LLE로 학습합니다.
* 건조기 필터가 막히거나, 모터에 이상이 생기거나, 센서가 고장 나면 데이터가 이 정상적인 형태에서 벗어나게 되므로 이를 감지하여 고장을 사전에 예방할 수 있습니다.

### 건조 상태 모니터링 및 시각화

* 수많은 센서에서 수집되는 다차원 데이터를 2차원이나 3차원으로 축소하여 차트로 시각화합니다.
* 데이터가 이동하는 궤적을 보면서 현재 건조기가 '초기 가열 단계'인지, '본격 건조 단계'인지, '쿨링 단계'인지 한눈에 파악할 수 있습니다.

### 지능형 건조 종료 시간(AI 남은 시간) 예측

* 입력되는 복합 센서 변수가 너무 많으면 머신러닝 모델이 무거워지고 과적합(Overfitting)이 발생할 수 있습니다. LLE로 핵심적인 데이터 구조만 압축하여 가벼운 예측 모델의 입력값으로 사용합니다. (MDPI)


## 2. 건조기 데이터에 LLE를 적용할 때의 꿀팁 (전처리 방법)

### 변수 간 스케일 맞추기 (필수)

* 온도는 $30$ ~ $70^oC$, 전류는 0 ~ 15**A**, 진동은 소수점 단위 등 센서마다 데이터의 단위와 범위가 완전히 다릅니다.
* LLE는 거리 기반 알고리즘이므로, 반드시 StandardScaler나 MinMaxScaler를 이용해 모든 센서 데이터의 범위를 통일해 주어야 특정 센서에만 결과가 좌우되는 것을 막을 수 있습니다.

### 시간 축을 고려한 윈도우(Window) 기법 적용

* 특정 '시점'의 센서 값만 넣으면 전후 흐름을 놓칩니다.
* [현재 온도, 현재 습도, 현재 진동]만 넣는 것이 아니라, [5초 전 온도, 5초 전 습도, ... , 현재 온도, 현재 습도] 처럼 일정 시간 동안의 센서 묶음(Time Window)을 하나의 고차원 벡터로 펴서 LLE에 입력하는 것이 훨씬 효과적입니다.

### 💡 다른 알고리즘과의 비교 및 추천

* 데이터에 노이즈(튀는 값)가 적고 비교적 정교한 물리적 궤적을 찾고 싶다면 LLE나 Isomap이 좋습니다.
* 만약 센서 데이터의 노이즈가 너무 심하다면 LLE 대신 안정적인 PCA(주성분 분석)를 먼저 적용해 보거나, 최근 가전/산업 AI에서 가장 많이 쓰이는 오토인코더(Autoencoder) 기반의 딥러닝 차원 축소를 고려해 보시는 것을 권장합니다. (MDPI)

## 3. 단일 클래스 분류

건조기 센서 데이터를 이용해 고장을 진단할 때는 정상 상태 데이터만으로 정상 범위를 정의하고, 이를 벗어나는 데이터를 '고장'으로 분류하는 단일 클래스 분류(Semi-supervised Anomaly Detection) 기법이 실무에서 가장 많이 쓰입니다.

LLE로 센서들의 복잡한 비선형 관계를 2차원으로 축소한 뒤, 이상 탐지 모델인 One-Class SVM을 적용하는 파이썬 예제 코드입니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

# 1. 가상의 건조기 복합 센서 데이터 생성 (정상 데이터 200개, 고장 데이터 20개)
np.random.seed(42)

# [정상] 온도, 습도, 진동, 전류가 서로 복잡한 비선형 관계를 가지며 순환함
n_normal = 200
t = np.linspace(0, 4 * np.pi, n_normal)
normal_temp = 50 + 20 * np.sin(t) + np.random.normal(0, 1.5, n_normal)
normal_humid = 55 + 35 * np.cos(t) + np.random.normal(0, 1.5, n_normal)
normal_vib = 0.3 + 0.1 * np.sin(t*2) + np.random.normal(0, 0.02, n_normal)
normal_curr = 8 + 3 * np.cos(t/2) + np.random.normal(0, 0.2, n_normal)
X_normal = np.column_stack([normal_temp, normal_humid, normal_vib, normal_curr])

# [고장] 필터 막힘이나 과열 등으로 인해 정상 궤적을 벗어난 이상치 생성
n_fault = 20
fault_temp = np.random.uniform(75, 85, n_fault)
fault_humid = np.random.uniform(60, 70, n_fault)
fault_vib = np.random.uniform(0.4, 0.6, n_fault)
fault_curr = np.random.uniform(10, 13, n_fault)
X_fault = np.column_stack([fault_temp, fault_humid, fault_vib, fault_curr])

# 전체 데이터 병합 (220개 데이터 포인트)
X_all = np.vstack([X_normal, X_fault])
y_true = np.array([1] * n_normal + [-1] * n_fault) # 1: 정상, -1: 고장


# 2. 데이터 전처리 (Standard Scaling)
# LLE는 거리 기반이므로 변수들의 스케일(단위)을 반드시 맞춰야 합니다.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)


# 3. LLE를 이용한 차원 축소 (4차원 -> 2차원)
# n_neighbors는 이웃의 수로, 데이터 밀도에 맞게 튜닝이 필요합니다.
lle = LocallyLinearEmbedding(n_neighbors=15, n_components=2, random_state=42)
X_lle = lle.fit_transform(X_scaled)


# 4. 고장 진단 모델 적용 (One-Class SVM)
# 실무에서는 '정상 데이터'만 가지고 학습(fit)을 수행하여 정상 범위를 규정합니다.
X_train_normal = X_lle[:n_normal]
oc_svm = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale')
oc_svm.fit(X_train_normal)

# 전체 데이터(정상+고장)에 대해 고장 여부 예측 (1: 정상, -1: 고장)
y_pred = oc_svm.predict(X_lle)


# 5. 시각화 (2차원 평면에 정상과 고장 표현)
plt.figure(figsize=(10, 6))

# 실제 레이블을 기준으로 시각화
plt.scatter(X_lle[:n_normal, 0], X_lle[:n_normal, 1], 
            c='blue', label='Normal (True)', alpha=0.6, edgecolor='k')
plt.scatter(X_lle[n_normal:, 0], X_lle[n_normal:, 1], 
            c='red', label='Fault (True)', alpha=0.8, edgecolor='k', marker='X', s=100)

plt.title('Dryer Fault Detection using LLE and One-Class SVM')
plt.xlabel('LLE Component 1')
plt.ylabel('LLE Component 2')
plt.legend()
plt.grid(True)
plt.show()

# 6. 간단한 평가 결과 출력
n_false_positive = np.sum((y_true[:n_normal] == 1) & (y_pred[:n_normal] == -1))
n_true_positive_fault = np.sum((y_true[n_normal:] == -1) & (y_pred[n_normal:] == -1))

print(f"[진단 결과]")
print(f"총 정상 데이터 {n_normal}개 중 {n_false_positive}개를 고장으로 오탐(False Alarm)")
print(f"총 고장 데이터 {n_fault}개 중 {n_true_positive_fault}개를 정확히 탐지")


### 🔍 코드의 핵심 포인트

* 학습용 데이터 분리:
  * 머신러닝을 이용한 설비 고장 진단에서는 고장 데이터가 매우 드물게 발생합니다.
  * 따라서 X_train_normal처럼 정상 데이터로만 모델을 가르치고, 완성된 모델이 고장 데이터를 걸러내도록 설계해야 합니다.
* n_neighbors 하이퍼파라미터:
  * LLE에서 가장 중요한 옵션입니다.
  * 값이 너무 작으면 데이터의 연속성이 끊어지고, 너무 크면 비선형적 굴곡을 무시해 버립니다.
  * 데이터셋의 크기에 맞게 조절해 보세요.
* One-Class SVM의 nu 옵션:
  * nu=0.05는 학습에 쓰인 정상 데이터 중 약 5%는 이상치(노이즈)로 취급하여 무시하겠다는 뜻입니다.
  * 현장의 노이즈 수준에 따라 이 값을 조절하여 오탐률을 제어할 수 있습니다.

## 4. lle에서 정상치 만으로 고장 여부를 정의할 수 있을까?

네, 가능합니다. 엄밀히 말하면 LLE 단독으로는 불가능하지만, LLE로 차원을 축소한 뒤 정상 데이터만으로 학습하는 이상 탐지(Anomaly Detection) 알고리즘을 결합하면 완벽하게 구현할 수 있습니다.

실제로 산업 현장이나 가전제품 고장 진단에서는 고장 데이터가 매우 드물게 발생하기 때문에, "정상치 데이터만 사용하여 고장 여부를 판별하는 방식"이 표준적으로 사용됩니다.

이 방식이 가능한 이유와 구체적인 구현 메커니즘을 2가지 관점에서 설명해 드립니다.

### 4.1 매니폴드(Manifold) 관점: "정상의 형태" 정의하기

LLE는 데이터가 이루는 고유한 기하학적 공간(매니폴드)을 학습합니다. 

* 정상 데이터만 LLE에 입력하면 건조기가 정상 작동할 때 나타나는 온도, 습도, 전류 등의 유기적인 곡선 형태(정상 매니폴드)가 2차원이나 3차원 공간에 예쁘게 펼쳐집니다.
* 만약 고장이나 이상이 발생하여 센서 값들의 밸런스가 깨지면, 축소된 공간에서 정상 데이터들이 모여 있는 군집과 멀리 떨어진 곳에 데이터가 위치하게 됩니다.

### 4.2 구체적인 2가지 구현 방법

#### 방법 A. LLE + 단일 클래스 분류기 (가장 추천)

이전 답변에서 보여드린 코드의 방식입니다. 정상 데이터만 수집하여 LLE로 차원을 축소한 뒤, 그 좌표를 One-Class SVM이나 Isolation Forest 같은 알고리즘에 학습시킵니다.

* 작동 원리: 정상 데이터가 위치한 영역에만 '경계선'을 치고, 새로운 데이터가 들어왔을 때 이 경계선 안쪽에 있으면 정상, 바깥쪽에 있으면 고장으로 판단합니다.

#### 방법 B. LLE의 재구성 오차 (Reconstruction Error) 활용

LLE 알고리즘 자체의 원리를 이용하는 고급 기법입니다.

* 정상 데이터들만 가지고 주변 이웃들과의 가중치(*W*) 관계를 정교하게 학습해 둡니다.
* 새로운 센서 데이터가 들어왔을 때, 기존에 학습된 정상 데이터들의 가중치 조합으로 이 새로운 데이터를 복원(Reconstruct)해 봅니다.
* 작동 원리:
  * 들어온 데이터가 정상적인 흐름의 데이터라면 기존 가중치로 아주 잘 복원되지만, 고장 데이터라면 복원이 잘 되지 않고 큰 오차가 발생합니다.
  * 이 복원 오차(Reconstruction Error)가 특정 기준치(Threshold)를 넘으면 고장으로 진단합니다.

### ⚠️ 실무 적용 시 반드시 알아야 할 한계점

* Out-of-Sample 문제:
  * scikit-learn의 LLE는 기본적으로 새로운 데이터가 들어왔을 때 이를 저차원으로 변환하는 transform() 연산이 매우 무겁고 불안정할 수 있습니다.
  * 새로운 데이터가 들어올 때마다 주변 이웃을 다시 계산해야 하기 때문입니다.
* 해결책:
  * 실무에서 실시간 고장 진단을 할 때는 LLE 대신, 데이터의 압축과 복원 오차를 모두 제공하면서 실시간 연산이 매우 빠른 오토인코더(Autoencoder) 딥러닝 모델을 더 선인하는 편입니다. (ScienceDirect.com)

정상 데이터만 보유하고 계신 상황이라면 위 매커니즘대로 충분히 훌륭한 고장 진단 시스템의 초안을 만드실 수 있습니다.